# LearnMateAI — Candidate Adapter Evaluation

Scores `qwen25-lora-20260815-090709` on **both** held-out sets in one Colab GPU session, compares against a fallback API, and appends two rows to `version_registry.csv`.

This PC cannot run the live eval (no NVIDIA GPU, no local PyTorch). **Run this notebook in Colab on a T4.** This is evaluation, not a second training run.

## Golden path

1. Runtime → Disconnect and delete runtime (if you used this notebook before)
2. Runtime → Change runtime type → **T4 GPU**
3. Run **0 — Install** once, then **Runtime → Restart session**
4. Run **0b — Verify**, then CONFIG
5. When prompted, upload:
   - the adapter folder zip (`adapter_model.safetensors` + `adapter_config.json` + `run_record.json`)
   - `test.jsonl` and `test_strict.jsonl` from `01_dataset_pipeline/processed_v01/`
   - `acceptance_thresholds.yaml` and `version_registry.csv` from this folder
6. Put `OPENAI_API_KEY` or `GEMINI_API_KEY` in **Colab Secrets** (key icon) or a gitignored `.env` — never paste a key into a cell, chat, or `!export`. Fallback comparison is mandatory. Rotate any key that was ever pasted.
7. Run the rest top to bottom. Expect ~20–40 minutes for 325 + 280 items.
8. Download the updated `version_registry.csv` and the `eval_predictions/` folder

The two accuracy numbers are **not** interchangeable:
- `test.jsonl` → `in_corpus_accuracy (chapter-held-out)`
- `test_strict.jsonl` → `accuracy (document-held-out)` (closer to the 0.70 bar)

## 0 — Install (Colab)

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Do not touch torch / torchvision / pillow / numpy — same rule as the training notebook.
    %pip install -q -U transformers accelerate peft bitsandbytes sentencepiece pyyaml pandas openai python-dotenv
    print("Installed eval stack. NEXT: Runtime -> Restart session, then run 0b.")
else:
    print("Not Colab. Live eval needs a CUDA GPU; this Windows machine does not have one.")

print("IN_COLAB =", IN_COLAB)

## 0b — Verify environment (after Restart session)

In [ ]:
import importlib
import sys

IN_COLAB = "google.colab" in sys.modules

def _v(mod: str) -> str:
    try:
        m = importlib.import_module(mod)
        return getattr(m, "__version__", "ok")
    except Exception as exc:
        return f"IMPORT FAILED: {exc}"

import torch
print("torch         :", torch.__version__)
print("cuda          :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu           :", torch.cuda.get_device_name(0))
for _mod in ["transformers", "peft", "accelerate", "bitsandbytes", "openai", "yaml"]:
    print(f"{_mod:14s}:", _v(_mod))
if not torch.cuda.is_available():
    print("WARNING: no GPU. Do not run the live eval on CPU in Colab — switch to T4.")

## 1 — CONFIG

In [ ]:
from pathlib import Path

EVAL_CONFIG = {
    "adapter_dir": "adapter",
    "run_record_path": None,
    "thresholds_path": "acceptance_thresholds.yaml",
    "registry_path": "version_registry.csv",
    "predictions_dir": "eval_predictions",
    "max_new_tokens": 256,
    "temperature": 0.1,
    "dry_run": False,
    "candidate_id": None,
    # Preferred accuracy grader (Step 4). False keeps token-F1 / MCQ letter.
    # True uses the fallback API as LLM-as-judge — only after the exposed key is rotated.
    "use_llm_judge": False,
    "evals": [
        {
            "test_path": "data/test.jsonl",
            "metric_definition": "in_corpus_accuracy (chapter-held-out)",
        },
        {
            "test_path": "data/test_strict.jsonl",
            "metric_definition": "accuracy (document-held-out)",
        },
    ],
}

print(EVAL_CONFIG)

## 1b — Upload files (Colab) + API key

Upload, together:
- contents of `02_finetuning/adapters/qwen25-lora-20260815-090709/adapter/` (or a zip of that folder)
- `processed_v01/test.jsonl` and `processed_v01/test_strict.jsonl`
- `acceptance_thresholds.yaml` and `version_registry.csv`

Then set **one** fallback key via Colab Secrets or a gitignored `.env`: `OPENAI_API_KEY` or `GEMINI_API_KEY`. Never paste a live key into this notebook, a terminal argument, or chat.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

def _looks_like_adapter(d: Path) -> bool:
    return (d / "adapter_config.json").exists() and (
        (d / "adapter_model.safetensors").exists() or (d / "adapter_model.bin").exists()
    )

def _ensure_files() -> None:
    adapter = Path(EVAL_CONFIG["adapter_dir"])
    needed = [Path(e["test_path"]) for e in EVAL_CONFIG["evals"]]
    needed += [Path(EVAL_CONFIG["thresholds_path"]), Path(EVAL_CONFIG["registry_path"])]
    missing = [p for p in needed if not p.exists()] + ([] if _looks_like_adapter(adapter) else [adapter])
    if not missing:
        print("All eval files present.")
        return
    if not IN_COLAB:
        raise FileNotFoundError(
            "Missing: " + ", ".join(str(p) for p in missing) + ". Run this notebook in Colab and upload them."
        )
    print("Upload adapter files (or a .zip), test.jsonl, test_strict.jsonl, acceptance_thresholds.yaml, version_registry.csv")
    from google.colab import files
    uploaded = files.upload()
    Path("data").mkdir(exist_ok=True)
    adapter.mkdir(parents=True, exist_ok=True)
    for name, content in uploaded.items():
        dest = Path(name)
        dest.write_bytes(content)
        print("saved", dest, len(content), "bytes")
        if dest.suffix.lower() == ".zip":
            with zipfile.ZipFile(dest) as zf:
                zf.extractall("_unzipped")
            hits = [p.parent for p in Path("_unzipped").rglob("adapter_config.json")]
            if hits:
                if adapter.exists():
                    shutil.rmtree(adapter)
                shutil.copytree(hits[0], adapter)
                print("adapter extracted to", adapter)
        elif dest.name.endswith(".jsonl"):
            shutil.copy(dest, Path("data") / dest.name)
            print("copied jsonl to", Path("data") / dest.name)
        elif dest.name in {"acceptance_thresholds.yaml", "version_registry.csv"}:
            print("config/registry in cwd")
        elif dest.name in {"adapter_config.json", "adapter_model.safetensors", "run_record.json",
                           "tokenizer.json", "tokenizer_config.json", "chat_template.jinja"}:
            shutil.copy(dest, adapter / dest.name)

    still = [p for p in needed if not p.exists()]
    if not _looks_like_adapter(adapter):
        still.append(adapter)
    if still:
        raise FileNotFoundError("Still missing after upload: " + ", ".join(str(p) for p in still))

_ensure_files()

# Fallback key: Gemini preferred, OpenAI accepted (same OpenAI-compatible client).
if IN_COLAB and not (os.getenv("GEMINI_API_KEY") or os.getenv("OPENAI_API_KEY") or os.getenv("LM_API_KEY")):
    from google.colab import userdata
    for key in ("GEMINI_API_KEY", "OPENAI_API_KEY", "LM_API_KEY"):
        try:
            val = userdata.get(key)
        except Exception:
            val = None
        if val:
            os.environ[key] = val
            print("loaded", key, "from Colab secrets (value not printed)")
            break
    else:
        print("No API key in env or Colab secrets.")
        print("Add a secret named OPENAI_API_KEY or GEMINI_API_KEY (key icon in the left sidebar), then re-run this cell.")

print("adapter   :", Path(EVAL_CONFIG["adapter_dir"]).resolve())
print("tests     :", [e["test_path"] for e in EVAL_CONFIG["evals"]])

## 2 — Load thresholds + run-record

In [ ]:
import json
from pathlib import Path
import yaml

with open(EVAL_CONFIG["thresholds_path"], encoding="utf-8") as f:
    THRESHOLDS = yaml.safe_load(f)

adapter_dir = Path(EVAL_CONFIG["adapter_dir"])
run_record_path = Path(EVAL_CONFIG["run_record_path"] or (adapter_dir / "run_record.json"))
assert run_record_path.exists(), f"Missing run-record at {run_record_path}"
with run_record_path.open(encoding="utf-8") as f:
    RUN_RECORD = json.load(f)

assert RUN_RECORD.get("dataset_version") == "lm-legal-v0.1", RUN_RECORD.get("dataset_version")
CANDIDATE_ID = EVAL_CONFIG["candidate_id"] or RUN_RECORD["run_id"]
print("candidate:", CANDIDATE_ID)
print("dataset  :", RUN_RECORD["dataset_version"])
print("train/val:", RUN_RECORD.get("train_examples"), RUN_RECORD.get("val_examples"))
RUN_RECORD

## 3 — Metric helpers

In [ ]:
import json
import re
import time
from collections import Counter

# Same citation checker as Stage 2 (validate_pairs.py). The old string-set
# regex treated "108." in the excerpt as unrelated to "section 108" in the
# answer and over-flagged ~40-50% of the first live eval.
_CITE = re.compile(
    r"\b(?:section|sections|sec\.|s\.|article|articles|art\.|chapter|part|rule|order)\s+"
    r"([0-9]{1,3}[A-Za-z]{0,2})",
    re.I,
)
_IN_SRC = (
    re.compile(
        r"\b(?:section|sections|sec\.|s\.|article|articles|art\.|chapter|part|rule|order)\s+"
        r"([0-9]{1,3}[A-Za-z]{0,2})",
        re.I,
    ),
    re.compile(r"(?:^|\n)\s*([0-9]{1,3}[A-Za-z]{0,2})\s*\."),
    re.compile(r"([0-9]{1,3}[A-Za-z]{0,2})\s*\.\s*\([0-9a-z]"),
    re.compile(r"\b([0-9]{1,3}[A-Za-z]{1,2})\b"),
)

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def tokenize(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())

def token_f1(pred: str, gold: str) -> float:
    p, g = tokenize(pred), tokenize(gold)
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0
    pc, gc = Counter(p), Counter(g)
    overlap = sum((pc & gc).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(g)
    return 2 * precision * recall / (precision + recall)

def mcq_letter(text: str):
    m = re.search(r"\b([ABCD])\b", text.upper())
    return m.group(1) if m else None

JUDGE_PROMPT = (
    "You grade a study-assistant answer against a gold answer AND a source excerpt. "
    "The live product treats any claim not supported by retrieved context as a hallucination. "
    "Score CORRECT only if the candidate is substantively right according to the gold/excerpt "
    "and does not invent section numbers, facts, or holdings absent from the excerpt. "
    'Return JSON: {"correct": true or false, "reason": "one sentence"}.'
)

def llm_judge_correct(pred: str, gold: str, source: str) -> bool:
    resp = client.chat.completions.create(
        model=EVAL_CONFIG.get("judge_model") or FALLBACK_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {
                "role": "user",
                "content": (
                    f"SOURCE EXCERPT:\n{source[:4000]}\n\n"
                    f"GOLD:\n{gold[:1500]}\n\nCANDIDATE:\n{pred[:1500]}"
                ),
            },
        ],
    )
    raw = resp.choices[0].message.content or "{}"
    try:
        return bool(json.loads(raw).get("correct"))
    except json.JSONDecodeError:
        return False

def is_correct(pair_type: str, pred: str, gold: str, f1_thresh: float, source: str = "") -> bool:
    if EVAL_CONFIG.get("use_llm_judge"):
        return llm_judge_correct(pred, gold, source)
    if pair_type == "mcq":
        return mcq_letter(pred) is not None and mcq_letter(pred) == mcq_letter(gold)
    return token_f1(pred, gold) >= f1_thresh

def numbers_in_excerpt(text: str) -> set:
    found = set()
    for rx in _IN_SRC:
        for m in rx.finditer(text):
            found.add(m.group(1).lower())
    return found

def hallucination_flag(pred: str, source: str, section_id=None) -> bool:
    answer = (pred or "").strip()
    if len(answer) < 40:
        return True
    cited = {m.group(1).lower() for m in _CITE.finditer(answer)}
    if not cited:
        return False
    permitted = numbers_in_excerpt(source or "")
    if section_id:
        permitted.add(str(section_id).lower())
    return len(cited - permitted) > 0

def extract_user_and_gold(row):
    msgs = row["messages"]
    user = next(m["content"] for m in msgs if m["role"] == "user")
    gold = next(m["content"] for m in msgs if m["role"] == "assistant")
    source = user.split("---SOURCE EXCERPT---")[-1].strip() if "---SOURCE EXCERPT---" in user else user
    return user, gold, source

def score_predictions(preds, rows, f1_thresh):
    correct = halluc = 0
    latencies = []
    for (pred, latency), row in zip(preds, rows):
        user, gold, source = extract_user_and_gold(row)
        correct += int(is_correct(row["pair_type"], pred, gold, f1_thresh, source))
        halluc += int(hallucination_flag(pred, source, row.get("section_id")))
        latencies.append(latency)
    n = len(rows)
    hall_rate = halluc / n
    p95 = sorted(latencies)[max(0, int(0.95 * (n - 1)))] if latencies else float("nan")
    return {
        "accuracy": correct / n,
        "groundedness": 1.0 - hall_rate,
        "hallucination_rate": hall_rate,
        "latency_p95_ms": p95,
        "n": n,
    }

print("metric helpers ready")

## 4 — Load candidate (QLoRA on T4)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

assert torch.cuda.is_available(), "Switch Runtime to T4 GPU, then Restart session."

base_id = RUN_RECORD["base_model_id"]
tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
base = AutoModelForCausalLM.from_pretrained(
    base_id,
    quantization_config=bnb,
    device_map={ "": 0 },
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(base, str(adapter_dir))
model.eval()

def candidate_generate(user_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are LearnMateAI, a study assistant for Sri Lankan legal education. "
                "Answer from the provided source excerpt. If the excerpt is insufficient, say so."
            ),
        },
        {"role": "user", "content": user_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=EVAL_CONFIG["max_new_tokens"],
            temperature=EVAL_CONFIG["temperature"],
            do_sample=EVAL_CONFIG["temperature"] > 0,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

print("candidate ready:", base_id)

## 5 — Fallback client (Gemini or OpenAI)

In [ ]:
import os
from openai import OpenAI

fb = THRESHOLDS["metrics"]["fallback_comparison"]
gemini_key = os.getenv(fb["fallback_env_api_key"])
openai_key = os.getenv("OPENAI_API_KEY") or os.getenv("LM_API_KEY")

if gemini_key:
    FALLBACK_MODEL = os.getenv(fb["fallback_env_model"], fb["default_fallback_model"])
    client = OpenAI(
        api_key=gemini_key,
        base_url=os.getenv("LM_API_BASE") or "https://generativelanguage.googleapis.com/v1beta/openai/",
    )
elif openai_key:
    FALLBACK_MODEL = os.getenv("OPENAI_MODEL") or os.getenv("LM_MODEL") or "gpt-4o-mini"
    client = OpenAI(
        api_key=openai_key,
        base_url=os.getenv("LM_API_BASE") or "https://api.openai.com/v1",
    )
else:
    raise RuntimeError(
        "Fallback comparison is mandatory. Set GEMINI_API_KEY or OPENAI_API_KEY "
        "(Colab: Secrets → add the name → re-run the upload/key cell)."
    )

def fallback_generate(user_text: str) -> str:
    resp = client.chat.completions.create(
        model=FALLBACK_MODEL,
        temperature=0.1,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a study assistant for Sri Lankan legal education. "
                    "Answer only from the provided source excerpt."
                ),
            },
            {"role": "user", "content": user_text},
        ],
    )
    return (resp.choices[0].message.content or "").strip()

print("fallback ready:", FALLBACK_MODEL)

## 6 — Run both evals + write registry

In [ ]:
import csv
from datetime import datetime, timezone
from pathlib import Path

f1_thresh = THRESHOLDS["metrics"]["accuracy"]["f1_pass_threshold"]
pred_dir = Path(EVAL_CONFIG["predictions_dir"])
pred_dir.mkdir(exist_ok=True)
registry_path = Path(EVAL_CONFIG["registry_path"])
fieldnames = [
    "candidate_id", "run_id", "base_model", "dataset_version", "evaluated_at_utc",
    "eval_split", "metric_definition",
    "accuracy", "groundedness", "hallucination_rate", "latency_p95_ms",
    "fallback_model", "fallback_accuracy", "fallback_groundedness",
    "passed", "fail_reasons", "notes",
]

def decide(candidate_metrics, fallback_metrics):
    m = THRESHOLDS["metrics"]
    reasons = []
    if candidate_metrics["accuracy"] < m["accuracy"]["minimum"]:
        reasons.append(f"accuracy {candidate_metrics['accuracy']:.3f} < {m['accuracy']['minimum']}")
    if candidate_metrics["groundedness"] < m["groundedness"]["minimum"]:
        reasons.append(f"groundedness {candidate_metrics['groundedness']:.3f} < {m['groundedness']['minimum']}")
    if candidate_metrics["hallucination_rate"] > m["hallucination_rate"]["maximum"]:
        reasons.append(
            f"hallucination_rate {candidate_metrics['hallucination_rate']:.3f} > {m['hallucination_rate']['maximum']}"
        )
    if candidate_metrics["latency_p95_ms"] > m["latency_p95_ms"]["maximum_ms"]:
        reasons.append(
            f"latency_p95_ms {candidate_metrics['latency_p95_ms']:.0f} > {m['latency_p95_ms']['maximum_ms']} "
            f"(eval_hardware={m['latency_p95_ms'].get('eval_hardware')})"
        )
    slack = m["fallback_comparison"]["accuracy_slack"]
    ok = (
        candidate_metrics["accuracy"] >= fallback_metrics["accuracy"]
        or (
            candidate_metrics["groundedness"] >= fallback_metrics["groundedness"]
            and candidate_metrics["accuracy"] >= fallback_metrics["accuracy"] - slack
        )
    )
    if not ok:
        reasons.append(
            "failed fallback comparison "
            f"(cand_acc={candidate_metrics['accuracy']:.3f}, fb_acc={fallback_metrics['accuracy']:.3f}, "
            f"cand_ground={candidate_metrics['groundedness']:.3f}, fb_ground={fallback_metrics['groundedness']:.3f})"
        )
    return len(reasons) == 0, reasons

results = []
for spec in EVAL_CONFIG["evals"]:
    rows = load_jsonl(spec["test_path"])
    print(f"\n=== {spec['metric_definition']}  n={len(rows)}  file={spec['test_path']} ===")
    cand_preds, fb_preds = [], []
    dump = []
    for i, row in enumerate(rows, start=1):
        user, gold, source = extract_user_and_gold(row)
        t0 = time.perf_counter()
        pred = candidate_generate(user)
        dt = (time.perf_counter() - t0) * 1000
        cand_preds.append((pred, dt))
        t1 = time.perf_counter()
        fb_pred = fallback_generate(user)
        fb_dt = (time.perf_counter() - t1) * 1000
        fb_preds.append((fb_pred, fb_dt))
        dump.append({
            "pair_id": row.get("pair_id"),
            "subject_area": row.get("subject_area"),
            "pair_type": row.get("pair_type"),
            "gold": gold,
            "candidate": pred,
            "fallback": fb_pred,
            "candidate_ms": round(dt, 1),
            "fallback_ms": round(fb_dt, 1),
        })
        if i % 25 == 0 or i == len(rows):
            print(f"  {i}/{len(rows)}")

    split_name = Path(spec["test_path"]).stem
    with (pred_dir / f"{CANDIDATE_ID}_{split_name}.jsonl").open("w", encoding="utf-8") as f:
        for item in dump:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    candidate_metrics = score_predictions(cand_preds, rows, f1_thresh)
    fallback_metrics = score_predictions(fb_preds, rows, f1_thresh)
    passed, fail_reasons = decide(candidate_metrics, fallback_metrics)
    print("candidate:", candidate_metrics)
    print("fallback :", fallback_metrics)
    print("PASSED" if passed else "FAILED")
    for r in fail_reasons:
        print(" -", r)

    rec = {
        "candidate_id": CANDIDATE_ID,
        "run_id": RUN_RECORD.get("run_id", CANDIDATE_ID),
        "base_model": RUN_RECORD.get("base_model_id", ""),
        "dataset_version": RUN_RECORD.get("dataset_version", ""),
        "evaluated_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "eval_split": split_name,
        "metric_definition": spec["metric_definition"],
        "accuracy": f"{candidate_metrics['accuracy']:.4f}",
        "groundedness": f"{candidate_metrics['groundedness']:.4f}",
        "hallucination_rate": f"{candidate_metrics['hallucination_rate']:.4f}",
        "latency_p95_ms": f"{candidate_metrics['latency_p95_ms']:.1f}",
        "fallback_model": FALLBACK_MODEL,
        "fallback_accuracy": f"{fallback_metrics['accuracy']:.4f}",
        "fallback_groundedness": f"{fallback_metrics['groundedness']:.4f}",
        "passed": str(passed),
        "fail_reasons": " | ".join(fail_reasons),
        "notes": (
            f"grader={'llm_judge' if EVAL_CONFIG.get('use_llm_judge') else 'token-F1/MCQ'}; "
            "groundedness=validate_pairs; "
            f"eval_hardware={THRESHOLDS['metrics']['latency_p95_ms'].get('eval_hardware')}"
        ),
    }
    text = registry_path.read_text(encoding="utf-8") if registry_path.exists() else ""
    with registry_path.open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if text.strip() == "":
            writer.writeheader()
        writer.writerow(rec)
    results.append(rec)

print("\n=== SUMMARY ===")
for rec in results:
    print(rec["eval_split"], rec["metric_definition"], "passed=" + rec["passed"],
          "acc=" + rec["accuracy"], "ground=" + rec["groundedness"])
print("wrote", registry_path.resolve())

## 7 — Download results

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(EVAL_CONFIG["registry_path"])
    import shutil
    zip_path = shutil.make_archive("eval_predictions", "zip", EVAL_CONFIG["predictions_dir"])
    files.download(zip_path)
    print("Download the registry CSV into 03_testing_and_versioning/version_registry.csv")
    print("Do not start a second training run until those two rows are in the repo.")
else:
    print("Registry already written locally.")

## After Colab

Replace `03_testing_and_versioning/version_registry.csv` with the downloaded file (keep the dry-run example row plus the new ones). Do not start a second training run from a FAIL without reading *which* metric failed:

| Failing metric | What to try |
|---|---|
| Accuracy | Prompt-template match with training; epochs / LoRA rank; confirm GI-001-fixed pairs were trained |
| Groundedness | If clustered in single-document subjects, that is a corpus-coverage gap — more epochs will not fix it |
| Latency | Quantization / serving stack / **target hardware** — a Colab T4 sequential number is not production |
| Doesn't beat fallback | Keep the API as primary on those subjects (FR-11 routing); do not force a 1.5B LoRA to replace gpt-4o-mini |

- Both `passed=True` on **document-held-out** (and checklist signed by someone else) → then staging. Chapter-split pass alone is not a promotion.
- Do not edit `acceptance_thresholds.yaml` to force a pass.